#### fuzzy matching

In [1]:
import pandas as pd
from rapidfuzz import fuzz, process

In [2]:
# load data
sas_db = pd.read_csv(f'../data/ild_schema.csv')  
digital_db = pd.read_csv(f'../data/digital_schema.csv')  

In [3]:
# table mapping
table_map={}
for sas_table in sas_db['table_name'].unique().tolist():
    digital_table= digital_db['table_name'].unique().tolist()
    match, score, _= process.extractOne(sas_table, digital_table, scorer=fuzz.token_sort_ratio)
    if score >=80:
        table_map[sas_table]=match
table_map

{'AssessmentComponents': 'assessment_component', 'Sessions': 'session'}

In [4]:
# column mapping
col_map={}
for sas_table, digital_table in table_map.items():
    cols_sas_db=sas_db[sas_db["table_name"]==sas_table]
    col_digital_db=digital_db[digital_db["table_name"]==digital_table]
    cm= {}
    for _, row in cols_sas_db.iterrows():
        c=row['column_name']
        match, score, _ =process.extractOne(c,col_digital_db['column_name'].tolist(), scorer=fuzz.token_sort_ratio)
        if score >= 40:
            cm[c]=match
    col_map[sas_table]=cm
col_map


{'AssessmentComponents': {'AssessmentComponentID': 'assessment_code',
  'AssessmentCode': 'assessment_code',
  'AssessmentVersionNumber': 'assessment_version',
  'ComponentID': 'component_id',
  'ComponentVersionNumber': 'component_name',
  'ComponentName': 'component_name',
  'ExaminedInd': 'component_id',
  'SchoolInd': 'component_id',
  'SplitComponentInd': 'component_id',
  'ReposInd': 'component_id',
  'RegressionTolerance': 'assessment_code',
  'ComponentClassID': 'component_name'},
 'Sessions': {'SessionID': 'session_id',
  'SessionYear': 'session_year',
  'SessionMonth': 'session_name',
  'SessionName': 'session_name',
  'SessionType': 'session_type',
  'ExamProcessingMode': 'session_id',
  'BatchSessionTemplateInd': 'session_name',
  'PreBusinessStreamInd': 'business_stream_id',
  'BusinessStreamID': 'business_stream_id',
  'LinearStreamOpenInd': 'business_stream_id'}}

In [10]:
sas_db.columns

Index(['table_schema', 'table_name', 'column_name', 'equivalent_digital_name',
       'data_type', 'length', 'format', 'label', 'is_column_used_in_ild'],
      dtype='object')

In [11]:
grouped_columns =sas_db.groupby(['table_schema', 'table_name'])['column_name'].apply(list).reset_index()

In [12]:
for _, row in grouped_columns.iterrows():
    schema = row['table_schema']
    table = row['table_name']
    columns = row['column_name']
    print(f"Schema: {schema}, Table: {table}")
    print("Columns:", columns)
    print("-" * 40)

Schema: DB_CA, Table: AwardScheduleEvents
Columns: ['AwardScheduleEventID', 'BusinessStreamID', 'SessionID', 'AssessmentCode', 'AssessmentVersionNumber', 'ComponentID', 'AwardCode', 'EventDate', 'Entries', 'Notes', 'ModifiedBy', 'ModifiedDatetime', 'ETLLoadIDCreated', 'ETLLoadIDLastUpdated', 'ETLLoadTimeCreated', 'ETLLoadTimeLastUpdated', 'IsDeletedFromSource']
----------------------------------------
Schema: DB_CA, Table: CandidateAssessmentEntries
Columns: ['UciNumber', 'SessionID', 'SessionPartitionKey', 'AssessmentCode', 'AssessmentVersionNumber', 'BusinessStreamID', 'OptionCode', 'CentreNumber', 'CandidateNumber', 'PreviousCandidateNumber', 'PreviousCentreNumber', 'PrivateCandidateIndicator', 'ETLLoadIDCreated', 'ETLLoadIDLastUpdated', 'ETLLoadTimeCreated', 'ETLLoadTimeLastUpdated', 'ModDatetime', 'IsDeletedFromSource', 'CampaignID', 'CentrePartCode', 'SchoolID', 'CreateDatetime']
----------------------------------------
Schema: DB_CA, Table: CandidateComponentMarkSet
Columns: ['C

#### embedding similarity

In [46]:
import pandas as pd, numpy as np
from sentence_transformers import SentenceTransformer
import re

In [138]:
# read metadata of sas schema and digital schema
sas_db = pd.read_csv(f'../data/ild_schema.csv')  
digital_db = pd.read_csv(f'../data/digital_schema.csv')  
# keep only table_name and column_name
source=sas_db[sas_db['is_column_used_in_ild']==1][['table_name', 'column_name']].astype(str).drop_duplicates()
target=digital_db[['table_name', 'column_name']].astype(str).drop_duplicates()

In [140]:
# preprocessing
# def camel_to_spaces(s):
#     # return re.sub(r'(?<!^)(?=[A-Z])', ' ', s)
#     return re.sub(r'(?<=[a-z0-9])(?=[A-Z][a-z])', ' ', s)

def camel_to_spaces(text: str) -> str:
    """
    Convert CamelCase string to space-separated words,
    keeping abbreviations (all-uppercase sequences) intact.
    Special handling to keep 'IDs' together.
    """

    # First, protect the "IDs" pattern by temporarily replacing it
    text = re.sub(r'IDs', 'ID_sentinel_', text)

    # Insert space before capital letters followed by lowercase letters
    text = re.sub(r'(?<!^)(?=[A-Z][a-z])', ' ', text)

    # Insert space between lowercase followed by uppercase
    text = re.sub(r'([a-z])([A-Z])', r'\1 \2', text)

    # Restore "IDs"
    text = text.replace("ID_sentinel_", "IDs")

    # Normalize spaces
    return " ".join(text.split())

source['table_name_processed']=source['table_name'].apply(camel_to_spaces)
source['column_name_processed']=source['column_name'].apply(camel_to_spaces)

source['table_name_processed']=source['table_name_processed'].str.replace('_', ' ')
source['column_name_processed']=source['column_name_processed'].str.replace('_', ' ')

target['table_name_processed']=target['table_name'].str.replace('_', ' ')
target['column_name_processed']=target['column_name'].str.replace('_', ' ')

In [144]:
# use both table_name and column_name for calculating embeddings
source_text=(source.table_name_processed +" "+ source.column_name_processed).str.lower().tolist()
target_text=(target.table_name_processed +" "+ target.column_name_processed).str.lower().tolist()

In [148]:
# use both column_name for calculating embeddings
source_text=(source.column_name_processed).str.lower().tolist()
target_text=(target.column_name_processed).str.lower().tolist()

In [152]:
# open source embedding model from hugging face
# model=SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
model=SentenceTransformer("BAAI/bge-base-en-v1.5")
embedding_source=model.encode(source_text, normalize_embeddings=True)
embedding_target=model.encode(target_text, normalize_embeddings=True)
# cosine similarity
S= embedding_source @ embedding_target.T
topk=3
idx=np.argsort(-S, axis=1)[:,:topk]

for i, (source_table, source_column) in enumerate(source[['table_name', 'column_name']].to_numpy()):
    print(f"{source_table}.{source_column}->")
    for j in idx[i]:
        target_table,target_column=target.iloc[j][['table_name', 'column_name']].to_numpy()
        print(f"    {target_table}.{target_column} score={S[i,j]:.3f}")


AwardScheduleEvents.BusinessStreamID->
    centre.business_stream_id score=1.000
    qualification.business_stream_id score=1.000
    business_stream.business_stream_id score=1.000
AwardScheduleEvents.SessionID->
    candidate_entry.session_id score=1.000
    component_test.session_id score=1.000
    testlet_top_level.session_id score=1.000
AwardScheduleEvents.AssessmentCode->
    assessment_component.assessment_code score=1.000
    testlet_switch.assessment_code score=1.000
    assessment.assessment_code score=1.000
AwardScheduleEvents.ComponentID->
    timetable_groupsession.component_id score=1.000
    component_test.component_id score=1.000
    marker_availability.component_id score=1.000
AwardScheduleEvents.AwardCode->
    candidate_entry.assessment_code score=0.773
    component_test.assessment_code score=0.773
    timetable_groupsession.assessment_code score=0.773
AwardScheduleEvents.EventDate->
    testlet_activities.updated_date score=0.793
    testlet_top_level.updated_date s

In [225]:
i_idx=np.repeat(np.arange(len(source)), topk)
j_idx=idx.ravel()
ranks=np.tile(np.arange(1, topk+1), len(source))

matches=(
    pd.DataFrame({
        "source_table": source.table_name.values[i_idx],
        "source_column": source.column_name.values[i_idx],
        "target_table": target.table_name.values[j_idx],
        "target_column": target.column_name.values[j_idx],
        "score": S[i_idx,j_idx],
        "rank": ranks
    })
    # .sort_values(["source_table", "source_column", "rank"])
)

matches['score'] = matches['score'].round(3)

In [226]:
exact_matches=(
    matches[matches['score']>0.8]
    .groupby(['source_table', 'source_column'])
    .first()
    .reset_index()
)

In [227]:
exact_matches.shape

(82, 6)

In [228]:
exact_matches['source_column'].nunique()

30

In [229]:
exact_matches['target_column'].nunique()

22

In [230]:
exact_matches

,source_table,source_column,target_table,target_column,score,rank
0,AssessmentComponents,AssessmentCode,assessment_component,assessment_code,1.000,1
1,AssessmentComponents,AssessmentComponentID,session_assessment,session_assessment_id,0.845,1
2,AssessmentComponents,AssessmentVersionNumber,marking_candidate_entry,assessment_version,0.941,1
3,AssessmentComponents,BusinessStreamID,centre,business_stream_id,1.000,1
4,AssessmentComponents,ComponentID,timetable_groupsession,component_id,1.000,1
...,...,...,...,...,...,...
77,Sessions,SessionName,testlet_top_level,session_name,1.000,1
78,Sessions,SessionYear,session,session_year,1.000,1
79,vAggCandCMS,AggregationResultStatusID,aggregation_process_tracker,aggregation_status_id,0.953,1
80,vAggCandCMS,CandidateEntryID,custom_groupsession_centre_entry_mapping,candidate_entry_id,1.000,1
